In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2005_2006"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "DIQ_D.xpt"
csv_file = DATA_DIR / "DIQ_D_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Untouched source copy for final fidelity checking
source_df = df.copy()


print("=" * 80)
print("NHANES 2005-2006 DIQ_D XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "DIQ010",
    "DID040",
    "DIQ220",
    "DIQ160",
    "DIQ170",
    "DIQ180",
    "DIQ190A",
    "DIQ190B",
    "DIQ190C",
    "DIQ200A",
    "DIQ200B",
    "DIQ200C",
    "DIQ050",
    "DID060",
    "DIQ060U",
    "DID070",
    "DIQ230",
    "DIQ240",
    "DID250",
    "DID260",
    "DIQ260U",
    "DID270",
    "DIQ280",
    "DIQ290",
    "DIQ300S",
    "DIQ300D",
    "DID310S",
    "DID310D",
    "DID320",
    "DID330",
    "DID340",
    "DID350",
    "DIQ350U",
    "DIQ360",
    "DIQ080"
]


if len(df) != 9822:
    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 36:
    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


if df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "DIQ_D columns/order differ.\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if df.columns.duplicated().any():
    raise ValueError(
        "Duplicate column names found."
    )


print(
    "PASS: DIQ_D structure = "
    "9,822 rows x 36 columns."
)


# ============================================================
# 4. KNOWN SAS/XPORT TINY-ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "True numeric zeros:",
    f"{int(true_zero_counts.sum()):,}"
)

print(
    "Tiny artifact occurrences:",
    f"{int(tiny_counts.sum()):,}"
)


print("\nTiny values by variable:")

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 5. EXPECTED VALIDATED ZERO ARTIFACT COUNTS
# ============================================================

EXPECTED_TINY_COUNTS = {

    "DID250": 4,

    "DID260": 72,

    "DID270": 38,

    "DIQ300D": 1,

    "DID340": 137,

    "DID350": 71
}


EXPECTED_TINY_TOTAL = 323


# ============================================================
# 6. VALIDATE TINY ARTIFACT LOCATIONS
# ============================================================

actual_tiny_columns = set(
    tiny_counts[
        tiny_counts > 0
    ].index
)


expected_tiny_columns = set(
    EXPECTED_TINY_COUNTS.keys()
)


unexpected_columns = (
    actual_tiny_columns
    -
    expected_tiny_columns
)


if unexpected_columns:

    raise ValueError(
        "STOP: Tiny values found in unexpected "
        "DIQ_D variables:\n"
        + str(sorted(unexpected_columns))
    )


missing_expected_columns = (
    expected_tiny_columns
    -
    actual_tiny_columns
)


if missing_expected_columns:

    raise ValueError(
        "STOP: Expected tiny-value variables missing:\n"
        + str(sorted(missing_expected_columns))
    )


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    actual_count = int(
        tiny_counts[col]
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected {expected_count:,} tiny values, "
            f"found {actual_count:,}."
        )


if int(tiny_counts.sum()) != EXPECTED_TINY_TOTAL:

    raise ValueError(
        "Unexpected total tiny-value count."
    )


print(
    "\nPASS: Exactly 323 validated tiny artifacts found."
)


# ============================================================
# 7. RESTORE VALIDATED ZEROS
# ============================================================

print("\n--- ZERO RESTORATION ---")


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    count = int(
        mask.sum()
    )


    df.loc[
        mask,
        col
    ] = 0.0


    print(
        f"{col:<10}: "
        f"{count:,} tiny -> 0"
    )


# ============================================================
# 8. VERIFY RESTORED ZEROS
# ============================================================

print("\n--- RESTORED ZERO COUNTS ---")


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    print(
        f"{col:<10}: {zero_count:,}"
    )


    if zero_count != expected_count:

        raise ValueError(
            f"{col}: expected {expected_count:,} zeros, "
            f"found {zero_count:,}."
        )


print(
    "PASS: All 323 validated zeros restored."
)


# ============================================================
# 9. VERIFY NO TINY ARTIFACT REMAINS
# ============================================================

remaining_tiny = int(
    df.select_dtypes(
        include=[np.number]
    )
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nRemaining tiny artifacts:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny-value artifact remains."
    )


print(
    "PASS: No tiny-value artifacts remain."
)


# ============================================================
# 10. SEQN VALIDATION
# ============================================================

print("\n--- SEQN CHECK ---")


seqn_values = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn_values,
        np.round(seqn_values),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains fractional values."
    )


seqn_missing = int(
    df["SEQN"]
    .isna()
    .sum()
)


seqn_duplicates = int(
    df["SEQN"]
    .duplicated()
    .sum()
)


seqn_min = df["SEQN"].min()
seqn_max = df["SEQN"].max()


print(
    "Missing:",
    seqn_missing
)

print(
    "Duplicates:",
    seqn_duplicates
)

print(
    "Minimum:",
    seqn_min
)

print(
    "Maximum:",
    seqn_max
)


if seqn_missing != 0:
    raise ValueError(
        "Unexpected missing SEQN."
    )


if seqn_duplicates != 0:
    raise ValueError(
        "Duplicate SEQN values found."
    )


if seqn_min != 31128:
    raise ValueError(
        f"Unexpected minimum SEQN: {seqn_min}"
    )


if seqn_max != 41474:
    raise ValueError(
        f"Unexpected maximum SEQN: {seqn_max}"
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 11. IDENTIFY GENUINE DECIMAL VARIABLES
# ============================================================

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_counts = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:

        fractional_counts[col] = 0
        continue


    fractional = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    count = int(
        fractional.sum()
    )


    fractional_counts[col] = count


    print(
        f"{col:<10}: "
        f"fractional values = {count:,}"
    )


DECIMAL_COLUMNS = [
    col
    for col, count
    in fractional_counts.items()
    if count > 0
]


if DECIMAL_COLUMNS != ["DIQ280"]:

    raise ValueError(
        "Unexpected decimal-variable pattern.\n"
        f"Found: {DECIMAL_COLUMNS}"
    )


if fractional_counts["DIQ280"] != 28:

    raise ValueError(
        "Unexpected number of fractional DIQ280 values."
    )


print(
    "\nPASS: DIQ280 is the only genuine decimal variable."
)


# ============================================================
# 12. DIQ280 A1C VALIDATION
# ============================================================

print("\n--- DIQ280 A1C CHECK ---")


s = df["DIQ280"]


a1c_values = (
    s.between(
        5.0,
        15.0
    )
)


a1c_count = int(
    a1c_values.sum()
)


dont_know_count = int(
    s.eq(999)
    .sum()
)


missing_count = int(
    s.isna()
    .sum()
)


print(
    "A1C 5.0-15.0:",
    a1c_count
)

print(
    "Don't know 999:",
    dont_know_count
)

print(
    "Missing:",
    missing_count
)


if a1c_count != 128:
    raise ValueError(
        "Unexpected DIQ280 A1C response count."
    )


if dont_know_count != 242:
    raise ValueError(
        "Unexpected DIQ280 999 count."
    )


if missing_count != 9452:
    raise ValueError(
        "Unexpected DIQ280 missing count."
    )


print(
    "PASS: DIQ280 matches expected CDC/source structure."
)


# ============================================================
# 13. VALIDATE ZERO-CORRECTED VARIABLES
# ============================================================

print("\n--- VALIDATED ZERO FIELD CHECKS ---")


# DID250
s = df["DID250"]

if int(s.eq(0).sum()) != 4:
    raise ValueError(
        "DID250 zero count differs."
    )

print(
    "PASS: DID250 zeros = 4"
)


# DID260
s = df["DID260"]

if int(s.eq(0).sum()) != 72:
    raise ValueError(
        "DID260 zero count differs."
    )

print(
    "PASS: DID260 zeros = 72"
)


# DID270
s = df["DID270"]

if int(s.eq(0).sum()) != 38:
    raise ValueError(
        "DID270 zero count differs."
    )

print(
    "PASS: DID270 zeros = 38"
)


# DIQ300D
s = df["DIQ300D"]

if int(s.eq(0).sum()) != 1:
    raise ValueError(
        "DIQ300D zero count differs."
    )

print(
    "PASS: DIQ300D zeros = 1"
)


# DID340
s = df["DID340"]

if int(s.eq(0).sum()) != 137:
    raise ValueError(
        "DID340 zero count differs."
    )

print(
    "PASS: DID340 zeros = 137"
)


# DID350
s = df["DID350"]

if int(s.eq(0).sum()) != 71:
    raise ValueError(
        "DID350 zero count differs."
    )

print(
    "PASS: DID350 zeros = 71"
)


# ============================================================
# 14. EXPECTED SOURCE FOR FIDELITY
# ============================================================

#
# Expected source =
#
# original XPT
# +
# ONLY the six validated tiny -> zero changes
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0.0


# ============================================================
# 15. DEFINE INTEGER VS DECIMAL FIELDS
# ============================================================

#
# DIQ280 is genuine decimal A1C data.
#
# DO NOT:
#
# - convert DIQ280 to Int64
# - round DIQ280
# - trim its .0
# - apply float_format
#

DECIMAL_COLUMNS = [
    "DIQ280"
]


INTEGER_COLUMNS = [
    col
    for col in df.columns
    if col not in DECIMAL_COLUMNS
]


# ============================================================
# 16. CONFIRM INTEGER FIELDS ARE WHOLE NUMBERS
# ============================================================

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    fractional = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if int(
        fractional.sum()
    ) != 0:

        raise ValueError(
            f"STOP: {col} contains genuine decimal values."
        )


print(
    "\nPASS: All non-DIQ280 variables "
    "are exact whole-number fields."
)


# ============================================================
# 17. CLEAN SEQN + QUESTIONNAIRE CODE FORMATTING
# ============================================================

#
# Whole-number fields:
#
# 31128.0 -> 31128
# 1.0     -> 1
# 999.0   -> 999
#
# DIQ280 remains FLOAT:
#
# 6.0 -> 6.0
# 6.5 -> 6.5
#

for col in INTEGER_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "PASS: Whole-number questionnaire fields "
    "converted to clean integer formatting."
)

print(
    "PASS: DIQ280 remains untouched as float."
)


# ============================================================
# 18. VALUE FIDELITY BEFORE EXPORT
# ============================================================

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


pre_export_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    if col in INTEGER_COLUMNS:

        current_values = (
            df[col]
            .astype("Float64")
            .to_numpy(
                dtype=float,
                na_value=np.nan
            )
        )

    else:

        current_values = (
            df[col]
            .to_numpy()
        )


    if not np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(pre_export_failures)
    )


print(
    "PASS: DIQ_D values match expected XPT "
    "after validated zero restoration."
)


# ============================================================
# 19. EXPORT CSV
# ============================================================

#
# IMPORTANT:
#
# NO float_format
# NO round()
# NO decimal trimming
#
# DIQ280 must appear AS IS.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")
print(csv_file)


# ============================================================
# 20. RE-READ CSV WITH ROUND-TRIP PARSING
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 21. STRUCTURE AFTER EXPORT
# ============================================================

if csv_df.shape != expected_df.shape:

    raise ValueError(
        "CSV shape differs from XPT.\n"
        f"Expected: {expected_df.shape}\n"
        f"Actual:   {csv_df.shape}"
    )


if csv_df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "\nPASS: CSV structure matches XPT."
)


# ============================================================
# 22. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print("\n--- XPT -> CSV NUMERIC FIDELITY ---")


numeric_failures = []
total_numeric_differences = 0


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    equal = np.isclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    differences = int(
        (~equal).sum()
    )


    total_numeric_differences += (
        differences
    )


    if differences > 0:

        numeric_failures.append(
            (
                col,
                differences
            )
        )


print(
    "Total numeric differences:",
    total_numeric_differences
)


if numeric_failures:

    raise ValueError(
        "XPT -> CSV numeric differences found:\n"
        + str(numeric_failures)
    )


print(
    "PASS: All 36 CSV columns numerically "
    "match expected XPT values."
)


# ============================================================
# 23. MISSING-POSITION FIDELITY
# ============================================================

missing_mismatches = 0
missing_failures = []


for col in expected_df.columns:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    diff = int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


    missing_mismatches += diff


    if diff > 0:

        missing_failures.append(
            (
                col,
                diff
            )
        )


print(
    "\nMissing-position differences:",
    missing_mismatches
)


if missing_failures:

    raise ValueError(
        "Missing-value positions changed:\n"
        + str(missing_failures)
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 24. READ RAW CSV TOKENS
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    reader = csv.DictReader(f)
    raw_rows = list(reader)


if len(raw_rows) != 9822:

    raise ValueError(
        "Unexpected raw CSV row count."
    )


# ============================================================
# 25. STRICT RAW INTEGER FORMAT FIDELITY
# ============================================================

#
# Whole-number fields:
#
# ""              if missing
# str(int(value)) otherwise
#

raw_integer_mismatches = 0
raw_integer_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in INTEGER_COLUMNS:

        source_value = (
            expected_df.iloc[i][col]
        )


        expected_text = (
            ""
            if pd.isna(source_value)
            else str(
                int(source_value)
            )
        )


        actual_text = (
            row[col]
        )


        if actual_text != expected_text:

            raw_integer_mismatches += 1


            if len(
                raw_integer_examples
            ) < 20:

                raw_integer_examples.append(
                    {
                        "csv_row": i + 2,
                        "column": col,
                        "expected": expected_text,
                        "actual": actual_text
                    }
                )


print(
    "\nWhole-number raw-format differences:",
    raw_integer_mismatches
)


if raw_integer_mismatches != 0:

    print(
        "Examples:"
    )


    for example in raw_integer_examples:

        print(
            example
        )


    raise ValueError(
        "DIQ_D integer-field formatting differs."
    )


print(
    "PASS: Whole-number questionnaire fields "
    "have exact clean integer formatting."
)


# ============================================================
# 26. STRICT DIQ280 DECIMAL FIDELITY
# ============================================================

#
# DIQ280 must use the original decoded XPT representation:
#
# expected_text =
# str(float(source_value))
#
# Examples:
#
# 5.0   -> "5.0"
# 6.5   -> "6.5"
# 7.2   -> "7.2"
# 999.0 -> "999.0"
#

diq280_mismatches = 0
diq280_examples = []


for i, row in enumerate(
    raw_rows
):

    source_value = (
        expected_df.iloc[i]["DIQ280"]
    )


    expected_text = (
        ""
        if pd.isna(source_value)
        else str(
            float(source_value)
        )
    )


    actual_text = (
        row["DIQ280"]
    )


    if actual_text != expected_text:

        diq280_mismatches += 1


        if len(
            diq280_examples
        ) < 20:

            diq280_examples.append(
                {
                    "csv_row": i + 2,
                    "expected": expected_text,
                    "actual": actual_text
                }
            )


print(
    "DIQ280 raw decimal differences:",
    diq280_mismatches
)


if diq280_mismatches != 0:

    print(
        "Examples:"
    )


    for example in diq280_examples:

        print(
            example
        )


    raise ValueError(
        "DIQ280 decimal formatting changed."
    )


print(
    "PASS: DIQ280 appears AS IS in CSV."
)


# ============================================================
# 27. UNWANTED .0 CHECK IN INTEGER FIELDS
# ============================================================

dot_zero_count = 0
dot_zero_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col in INTEGER_COLUMNS:

        value = (
            row[col]
        )


        if (
            value != ""
            and
            value.endswith(".0")
        ):

            dot_zero_count += 1


            if len(
                dot_zero_examples
            ) < 20:

                dot_zero_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "\nUnwanted .0 in integer fields:",
    dot_zero_count
)


if dot_zero_count != 0:

    print(
        "Examples:",
        dot_zero_examples
    )

    raise ValueError(
        "Unnecessary .0 formatting remains "
        "in whole-number fields."
    )


print(
    "PASS: No unnecessary .0 "
    "in whole-number fields."
)


# ============================================================
# 28. REPORT LEGITIMATE DIQ280 .0 VALUES
# ============================================================

diq280_dot_zero = 0


for row in raw_rows:

    value = (
        row["DIQ280"]
    )


    if (
        value != ""
        and
        value.endswith(".0")
    ):

        diq280_dot_zero += 1


print(
    "\nLegitimate DIQ280 values ending '.0':",
    diq280_dot_zero
)


print(
    "PASS: DIQ280 .0 values are preserved "
    "because DIQ280 is a decimal A1C field."
)


# ============================================================
# 29. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0
scientific_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        lower = (
            value.lower()
        )


        if (
            "e-" in lower
            or
            "e+" in lower
        ):

            scientific_count += 1


            if len(
                scientific_examples
            ) < 20:

                scientific_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    print(
        "Examples:",
        scientific_examples
    )

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 30. FINAL TINY-ARTIFACT TEXT CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


tiny_string_count = (
    csv_text.count(
        "5.397605346934028e-79"
    )
)


print(
    "Tiny artifact strings in CSV:",
    tiny_string_count
)


if tiny_string_count != 0:

    raise ValueError(
        "Tiny XPORT artifact found in CSV."
    )


print(
    "PASS: No tiny artifact in CSV."
)


# ============================================================
# 31. FINAL ZERO CHECK AFTER EXPORT
# ============================================================

print("\n--- FINAL CSV ZERO CHECK ---")


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    actual_count = int(
        csv_df[col]
        .eq(0)
        .sum()
    )


    print(
        f"{col:<10}: "
        f"{actual_count:,}"
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: CSV zero count changed."
        )


print(
    "PASS: Final CSV zero restorations remain correct."
)


# ============================================================
# 32. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("DIQ_D CONVERSION COMPLETE")
print("=" * 80)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "\nValidated tiny -> zero:"
)

print(
    "DID250:",
    4
)

print(
    "DID260:",
    72
)

print(
    "DID270:",
    38
)

print(
    "DIQ300D:",
    1
)

print(
    "DID340:",
    137
)

print(
    "DID350:",
    71
)

print(
    "TOTAL:",
    EXPECTED_TINY_TOTAL
)

print(
    "\nGenuine decimal variables:",
    1
)

print(
    "DIQ280:",
    "PRESERVED AS IS"
)

print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)

print(
    "Missing-position differences:",
    missing_mismatches
)

print(
    "Whole-number raw differences:",
    raw_integer_mismatches
)

print(
    "DIQ280 raw differences:",
    diq280_mismatches
)

print(
    "Unwanted integer .0:",
    dot_zero_count
)

print(
    "Scientific notation:",
    scientific_count
)

print(
    "\nEXPECTED FINAL STATUS:"
)

print(
    "DIQ_D_corrected.csv = "
    "CORRECTED + FIDELITY CHECKED"
)

NHANES 2005-2006 DIQ_D XPT -> CSV
Rows: 9,822
Columns: 36
PASS: DIQ_D structure = 9,822 rows x 36 columns.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact occurrences: 323

Tiny values by variable:
DID340     137
DID260      72
DID350      71
DID270      38
DID250       4
DIQ300D      1

PASS: Exactly 323 validated tiny artifacts found.

--- ZERO RESTORATION ---
DID250    : 4 tiny -> 0
DID260    : 72 tiny -> 0
DID270    : 38 tiny -> 0
DIQ300D   : 1 tiny -> 0
DID340    : 137 tiny -> 0
DID350    : 71 tiny -> 0

--- RESTORED ZERO COUNTS ---
DID250    : 4
DID260    : 72
DID270    : 38
DIQ300D   : 1
DID340    : 137
DID350    : 71
PASS: All 323 validated zeros restored.

Remaining tiny artifacts: 0
PASS: No tiny-value artifacts remain.

--- SEQN CHECK ---
Missing: 0
Duplicates: 0
Minimum: 31128.0
Maximum: 41474.0
PASS: SEQN validated.

--- FRACTIONAL VALUE CHECK ---
SEQN      : fractional values = 0
DIQ010    : fractional values = 0
DID040    : fractional values = 0
DIQ2